In [5]:
import pandas as pd

In [9]:
# Carregando dados
df = pd.read_csv("/content/coffee-shop-sales-revenue.csv", sep="|")

# Verificando dados
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149116 entries, 0 to 149115
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    149116 non-null  int64  
 1   transaction_date  149116 non-null  object 
 2   transaction_time  149116 non-null  object 
 3   transaction_qty   149116 non-null  int64  
 4   store_id          149116 non-null  int64  
 5   store_location    149116 non-null  object 
 6   product_id        149116 non-null  int64  
 7   unit_price        149116 non-null  float64
 8   product_category  149116 non-null  object 
 9   product_type      149116 non-null  object 
 10  product_detail    149116 non-null  object 
dtypes: float64(1), int64(4), object(6)
memory usage: 12.5+ MB


Duas informações importantes analisando as informações do .info

**Não temos nenhuma linha vazia (149116 non-null)**

**Tenho que criar uma coluna de faturamento (uma métrica fundamental) fazendo a equação: Receita Bruta = Quantidade (transaction_qty) x Preço Unitário (unit_price)**

Percebo também que a coluna **transaction_date (data) está com o tipo object (texto). Para o Python ou o Power BI entenderem que isso é um calendário, preciso converter para o formato de data.**

In [12]:
# Criando uma coluna de faturamento (Receita Bruta = Preço Unitário * Quantidade)
df['faturamento'] = df['transaction_qty'] * df['unit_price']

# Convertendo a coluna de data (de texto para formato de data oficial do Python)
df['transaction_date'] = pd.to_datetime(df['transaction_date'])

#Re-exibindo as 5 primeiras linhas
df.head()

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,faturamento
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,6.0
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,9.0
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2.0
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.2


Para esse dataset e esse projeto, meu objetivo é demonstrar o conhecimento em outras ferramentas além do Python. Mais especificamente SQL e PowerBI. Portanto, para esse projeto específico, apenas fiz a limpeza de dados com o Python e agora vou exportar a tabela e abrir com o SQL

In [13]:
# Salvando a nossa tabela pronta em um novo arquivo CSV
# O comando index=False evita que o Python crie uma coluna inútil com a numeração das linhas
df.to_csv('vendas_cafeteria_limpas', index = False)

## **Análise Exploratória (SQL)**
Após a limpeza dos dados, exportamos a base para o Google BigQuery para responder às nossas perguntas de negócio.

1. Faturamento Total e Produtos Vendidos


```
SELECT
  SUM(transaction_qty) AS total_produtos_vendidos,
  SUM(faturamento) AS faturamento_total
FROM `portfolio-dados-rodrigo-lima.cafeteria_ny.vendas`
```

2. Produtos mais rentáveis no período da tarde (13h às 17h)


```
SELECT
  product_type AS tipo_produto,
  SUM(transaction_qty) AS total_vendido,
  ROUND(SUM(faturamento), 2) AS faturamento_tarde
FROM `portfolio-dados-rodrigo-lima.cafeteria_ny.vendas`
WHERE EXTRACT(HOUR FROM transaction_time) >= 13
  AND EXTRACT(HOUR FROM transaction_time) <= 17
GROUP BY
  tipo_produto
ORDER BY
  faturamento_tarde DESC
LIMIT 5
```






